# Splatastrophe reconstruction starter
Run runtime checks, then preview reconstruction commands. Install the GPU reconstruction environment from docs/DEVELOPMENT.md first. No experiment has been run or validated on GPU yet.


In [ ]:
from pathlib import Path
import platform
import sys
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())


## Locate or clone the repository
In VS Code, run the notebook from the cloned repository. In Colab, add a secret named `GITHUB_TOKEN` with read access to this repository, enable notebook access, then run the next cell. The token is supplied through a temporary askpass helper and is not stored in the remote URL. The checkout defaults to `/content/Splatastrophe`. A shared notebook does not share its runtime or credentials.


In [ ]:
import os
import subprocess
import tempfile

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").is_dir()
if IS_COLAB:
    REPO = Path("/content/Splatastrophe")
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").is_file()), None)
    if REPO is None:
        raise RuntimeError("Open this notebook from the Splatastrophe checkout.")

if not (REPO / ".git").is_dir():
    if not IS_COLAB:
        raise RuntimeError("Clone the repository first.")
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add the GITHUB_TOKEN secret with repository read access.")
    with tempfile.TemporaryDirectory() as temp_dir:
        helper = Path(temp_dir) / "askpass.py"
        helper.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["SPLAT_GITHUB_TOKEN"])\n')
        helper.chmod(0o700)
        child_env = dict(os.environ, GIT_ASKPASS=str(helper), GIT_TERMINAL_PROMPT="0", SPLAT_GITHUB_TOKEN=token)
        try:
            subprocess.run(["git", "-c", "credential.helper=", "clone", "https://github.com/SriyaKan/Splatastrophe.git", str(REPO)], env=child_env, check=True)
        finally:
            child_env.pop("SPLAT_GITHUB_TOKEN", None)
            del token
print("Repository:", REPO)


In [ ]:
subprocess.run([sys.executable, str(REPO / "scripts/check_runtime.py")], check=True)


## Experiment outputs
Use a unique run name. Large outputs are ignored by Git. Colab runtime storage is temporary: copy checkpoints and exports to persistent storage before ending a session. On Thunder, verify instance/storage lifecycle and save artifacts before stopping or deleting compute.


In [ ]:
from datetime import datetime, timezone
RUN_NAME = "runtime-check-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO / "outputs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", RUN_DIR)


## Reconstruction stages
Download Barn image frames from the official dataset and extract into `data/raw/barn/images`. Install Nerfstudio/COLMAP in this runtime before execution. Run one stage at a time. These cells preview commands until EXECUTE is explicitly enabled.


In [ ]:
EXECUTE = False  # Set True only after verifying environment, dataset and compute budget.
def stage(name, *extra):
    command = [sys.executable, str(REPO / "pipeline/reconstruction/run.py"), name, *extra]
    if EXECUTE:
        command.append("--execute")
    subprocess.run(command, cwd=REPO, check=True)


In [ ]:
stage("process")


In [ ]:
stage("split")


In [ ]:
stage("train")


## Evaluate and export
Set the exact config path printed by the successful training run. The evaluation set doubles as validation in this initial baseline: do not tune on it and claim an untouched test result. Pose estimation uses all views; appearance training excludes evaluation frames.


In [ ]:
TRAIN_CONFIG = ""  # e.g. outputs/barn/.../config.yml
if TRAIN_CONFIG:
    stage("eval", "--load-config", str(REPO / TRAIN_CONFIG))
    stage("export", "--load-config", str(REPO / TRAIN_CONFIG))
else:
    print("Set TRAIN_CONFIG after a successful GPU training run.")
